In [1]:
import os, shutil
import sys
os.makedirs("/data/train", exist_ok=True)

In [2]:
!{sys.executable} -m pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.0 MB/s  0:00:00


In [3]:
!git clone https://github.com/QwenLM/Qwen3-VL-Embedding.git

Cloning into 'Qwen3-VL-Embedding'...
remote: Enumerating objects: 472, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 472 (delta 39), reused 24 (delta 24), pack-reused 418 (from 2)
Receiving objects: 100% (472/472), 20.62 MiB | 29.13 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [15]:
!{sys.executable} -m pip install huggingface-hub
!hf download Qwen/Qwen3-VL-Embedding-2B --local-dir ./models/Qwen3-VL-Embedding-2B
!{sys.executable} -m pip install qwen-vl-utils
import sys
import os

/bin/bash: line 1: hf: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 169.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [qwen-vl-utils]0m [av]


In [12]:
%cd /Qwen3_VL_Embedding

/Qwen3_VL_Embedding


In [13]:
!{sys.executable} -m pip install munch

In [14]:
!{sys.executable} -m pip  -q install tqdm peft tensorboard 

In [16]:
import os
import shutil
import random
import gc

from peft import LoraConfig, get_peft_model
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from data_train import *
from itertools import cycle
from tqdm import tqdm
import numpy as np
from util_data import SUBSET_NAMES, TEMPLATES_SMALL

from Qwen3_VL_Embedding.src.models.qwen3_vl_embedding import Qwen3VLEmbedder

In [17]:

def fix_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_dataset_name_for_template(dataset):
    dataset_name = {
        "imagenet_100": "",
        "imagenet": "",
        "std10": "",
        "pets": "pet ",
        "fgvc_aircraft": "aircraft ",
        "cars": "car ",
        "eurosat": "satellite ",
        "dtd": "texture ",
        "flowers102": "flower ",
        "food101": "food ",
        "sun397": "scene ",
        "caltech101": "",
    }[dataset]
    return dataset_name

@torch.no_grad()
def get_mu_and_kappa(model, dataset):
    dataset_name = get_dataset_name_for_template(dataset)
    templates = TEMPLATES_SMALL

    all_mus =[]
    all_kappas =[]

    for class_name in SUBSET_NAMES[dataset]:
        class_texts =[]
        for template in templates:
            class_texts.append({"text": template.format(dataset_name, class_name) + "."})

        class_embs = model.process(class_texts)
        class_embs = F.normalize(class_embs, dim=-1)

        mu = class_embs.mean(dim=0)
        D = mu.shape[0]
        R = mu.norm()

        mu = mu / R
        kappa = (R * (D - R**2)) / torch.clamp(1 - R**2, min=1e-6)

        all_mus.append(mu)
        all_kappas.append(kappa)

    all_mus = torch.stack(all_mus)
    all_kappas = torch.stack(all_kappas)

    return all_mus, all_kappas

def get_image_embedding(model, images):
    image_inputs = [{"image": img} for img in images]
    image_embs = model.process(image_inputs)
    image_embs = F.normalize(image_embs, dim=-1)
    return image_embs


def get_acc(model, data_loader, mu, logit_scale, device):
    model.model.to(device)
    mu.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Evaluating"):
            labels = labels.to(device)
            image_embedding = get_image_embedding(model, images)
            # compute similarity and predict
            similarity = logit_scale * (image_embedding @ mu.T)
            preds = similarity.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def sample_tangent_gaussian(mu, kappa, num_samples, kappa_scale=0.05, kappa_max=500.0):
    C, D = mu.shape
    device = mu.device

    kappa = kappa * kappa_scale
    kappa = torch.clamp(kappa, min=1.0, max=kappa_max)

    eps = torch.randn((C, num_samples, D), device=device, dtype=mu.dtype)

    mu_expanded = mu.unsqueeze(1)
    dot_product = (eps * mu_expanded).sum(dim=-1, keepdim=True)
    eps = eps - dot_product * mu_expanded

    kappa_expanded = kappa.view(C, 1, 1)
    eps = eps / torch.sqrt(kappa_expanded)

    samples = mu_expanded + eps
    samples = F.normalize(samples, p=2, dim=-1)

    return samples

def build_text_distribution_samples(mu, kappa, num_samples=30, kappa_scale=0.05, kappa_max=500.0):
    C, D = mu.shape
    device = mu.device
    dtype = mu.dtype
    kappa = kappa * kappa_scale
    kappa = torch.clamp(kappa, min=1.0, max=kappa_max)
    eps = torch.randn((C, num_samples, D), device=device, dtype=dtype)
    mu_expanded = mu.unsqueeze(1)
    dot_product = (eps * mu_expanded).sum(dim=-1, keepdim=True)
    eps_tangent = eps - dot_product * mu_expanded
    kappa_expanded = kappa.view(C, 1, 1)
    eps_scaled = eps_tangent / torch.sqrt(kappa_expanded + 1e-6)
    samples = mu_expanded + eps_scaled
    samples = F.normalize(samples, p=2, dim=-1)

    return samples

def logit_from_h_vectorized(logit_scale, image_feats, centroids, area_index, chosen_centroids):
    logits_base = logit_scale * (image_feats @ centroids.t())
    logits_samples = logit_scale * (image_feats @ chosen_centroids.t())
    S = chosen_centroids.shape[0]
    logits_all = logits_base.unsqueeze(0).repeat(S, 1, 1)
    logits_all[:, :, area_index] = logits_samples.t()

    return logits_all

def compute_reg_vectorized(logit_scale, area_index, samples, feats_i, centroids):
    if feats_i.shape[0] == 0:
        return torch.tensor(0.0, device=feats_i.device)

    sampled_centroids = samples[area_index]
    logits_all = logit_from_h_vectorized(logit_scale, feats_i, centroids, area_index, sampled_centroids)
    logits_flat = logits_all.reshape(-1, logits_all.size(-1))
    labels_flat = torch.full((logits_flat.size(0),), area_index, device=feats_i.device, dtype=torch.long)

    return F.cross_entropy(logits_flat, labels_flat)

def train_one_epoch_update(
    model,
    opt_h,
    scaler,
    step,
    fewshot_train_loader,
    loader_iter_G,
    lamda1,
    lamda2,
    lamda3,
    writer,
    device,
    dataset="dtd",
    logit_scale=15
):
    model.model.train()

    for real_images, real_labels in tqdm(fewshot_train_loader):
        step += 1
        synth_images, synth_labels = next(loader_iter_G)
        real_labels = real_labels.to(device)
        synth_labels = synth_labels.to(device)
        mu, kappa = get_mu_and_kappa(model, dataset)
        torch.cuda.empty_cache()
        with torch.amp.autocast('cuda'):
            real_imgs_embedding = get_image_embedding(model, real_images)
            synth_imgs_embedding = get_image_embedding(model, synth_images)
            logits_real_all = logit_scale * (real_imgs_embedding @ mu.t())
            logits_synth_all = logit_scale * (synth_imgs_embedding @ mu.t())

            samples = build_text_distribution_samples(mu, kappa, num_samples=30)

        log_metrics = {"br": 0, "rr": 0, "bs": 0, "rs": 0}

        with torch.amp.autocast('cuda'):
            loss_real = F.cross_entropy(logits_real_all, real_labels)
            loss_synth = F.cross_entropy(logits_synth_all, synth_labels)

            total_loss = loss_real + lamda1 * loss_synth

            log_metrics["br"] = loss_real.item()
            log_metrics["bs"] = loss_synth.item()

        reg_losses =[]
        log_rr, log_rs = 0, 0

        present_classes = torch.cat([real_labels, synth_labels]).unique()
        number_of_classes = len(present_classes)

        with torch.amp.autocast('cuda'):
            for c_id_tensor in present_classes:
                c_id = c_id_tensor.item()

                r_idx = (real_labels == c_id).nonzero(as_tuple=True)[0]
                s_idx = (synth_labels == c_id).nonzero(as_tuple=True)[0]

                if len(r_idx) > 0:
                    l_reg_r = compute_reg_vectorized(
                        logit_scale=logit_scale,
                        area_index=c_id,
                        samples=samples,
                        feats_i=real_imgs_embedding[r_idx],
                        centroids=mu
                    )
                    reg_losses.append(lamda2 * l_reg_r / number_of_classes)
                    log_rr += l_reg_r.item() / number_of_classes

                if len(s_idx) > 0:
                    l_reg_s = compute_reg_vectorized(
                        logit_scale=logit_scale,
                        area_index=c_id,
                        samples=samples,
                        feats_i=synth_imgs_embedding[s_idx],
                        centroids=mu
                    )
                    reg_losses.append(lamda3 * l_reg_s / number_of_classes)
                    log_rs += l_reg_s.item() / number_of_classes

            if reg_losses:
                total_loss = total_loss + torch.sum(torch.stack(reg_losses))
        print(f'Total loss {total_loss.item()}')
        opt_h.zero_grad(set_to_none=True)
        scaler.scale(total_loss).backward()
        scaler.step(opt_h)
        scaler.update()

        writer.add_scalar("Loss_Batch/Real_Base", log_metrics["br"], step)
        writer.add_scalar("Loss_Batch/Real_Reg", log_rr, step)
        writer.add_scalar("Loss_Batch/Synth_Base", log_metrics["bs"], step)
        writer.add_scalar("Loss_Batch/Synth_Reg", log_rs, step)
        writer.add_scalar("Loss_Batch/Total", total_loss.item(), step)

        del total_loss, logits_real_all, logits_synth_all, samples

    return step